# 03 — DB2 on IBM Cloud: Ingestion & Validation
**PIX Fraud Intelligence | IBM Portfolio Project**

**Goals:**
- Connect to DB2 on IBM Cloud Lite via `ibm_db` driver
- Create the `TRANSACTIONS` table schema
- Ingest ~100k processed rows in batches
- Validate with SQL queries and read back into pandas

> **Setup:** Copy `config/credentials_template.json` to `config/credentials.json`  
> and fill in your DB2 on IBM Cloud Lite connection details.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.db2_connector import DB2Connector
import warnings
warnings.filterwarnings('ignore')

## 1. Connect to DB2 on IBM Cloud

In [2]:
conn = DB2Connector(credentials_path='../config/credentials.json')
print('Connected to DB2 on IBM Cloud!')

result = conn.query('SELECT CURRENT TIMESTAMP AS TS FROM SYSIBM.SYSDUMMY1')
print(f'DB2 server time: {result["TS"].iloc[0]}')

Connected to DB2 on IBM Cloud!


DB2 server time: 2026-06-15 18:18:43.784795


## 2. Load Processed Data

In [3]:
df = pd.read_parquet('../data/processed/transactions_features.parquet')
print(f'Shape: {df.shape} | Fraud rate: {df["fraude"].mean():.4%}')
df.head(3)

Shape: (100000, 13) | Fraud rate: 13.7350%


,valor_brl,saldo_anterior_pagador,saldo_anterior_recebedor,hora_dia,dia_util,horario_noturno,acima_limite_noturno,razao_saldo_residual,proporcao_valor_recebedor,tipo_transacao_enc,dia_semana_enc,fraude,split
0,1011935.75,4.990342e+06,5530.55,15.0,0.0,0.0,0.0,0.797221,0.994564,0.0,6.0,0,train
1,1293676.13,1.294970e+06,28564.82,12.0,0.0,0.0,0.0,0.000999,0.978397,0.0,0.0,0,train
2,1735989.58,1.297753e+07,25703.68,16.0,0.0,0.0,0.0,0.866231,0.985410,0.0,0.0,0,train


## 3. Create Table Schema
We define a typed schema based on the DataFrame columns.

In [4]:
try:
    conn.execute('DROP TABLE TRANSACTIONS')
    print('Existing table dropped.')
except Exception:
    print('Table did not exist, creating fresh.')

def pandas_dtype_to_db2(series: pd.Series) -> str:
    if pd.api.types.is_bool_dtype(series):
        return 'SMALLINT'
    elif pd.api.types.is_integer_dtype(series):
        return 'INTEGER'
    elif pd.api.types.is_float_dtype(series):
        return 'DOUBLE'
    else:
        return 'VARCHAR(32)'

col_defs = [
    f'  {col.upper()} {pandas_dtype_to_db2(df[col])}'
    for col in df.columns
]
ddl = 'CREATE TABLE TRANSACTIONS (\n' + ',\n'.join(col_defs) + '\n)'
print(ddl)

conn.execute(ddl)
print('\nTable TRANSACTIONS created.')

Existing table dropped.
CREATE TABLE TRANSACTIONS (
  VALOR_BRL DOUBLE,
  SALDO_ANTERIOR_PAGADOR DOUBLE,
  SALDO_ANTERIOR_RECEBEDOR DOUBLE,
  HORA_DIA DOUBLE,
  DIA_UTIL DOUBLE,
  HORARIO_NOTURNO DOUBLE,
  ACIMA_LIMITE_NOTURNO DOUBLE,
  RAZAO_SALDO_RESIDUAL DOUBLE,
  PROPORCAO_VALOR_RECEBEDOR DOUBLE,
  TIPO_TRANSACAO_ENC DOUBLE,
  DIA_SEMANA_ENC DOUBLE,
  FRAUDE INTEGER,
  SPLIT VARCHAR(32)
)



Table TRANSACTIONS created.


## 4. Batch Insert

In [5]:
df_db2 = df.rename(columns=lambda c: c.upper())

print(f'Inserting {len(df_db2):,} rows into TRANSACTIONS...')
conn.insert_dataframe(df_db2, table='TRANSACTIONS', batch_size=2000)
print('\nIngestion complete!')

Inserting 100,000 rows into TRANSACTIONS...


  Inserted rows 0 – 2000


  Inserted rows 2000 – 4000


  Inserted rows 4000 – 6000


  Inserted rows 6000 – 8000


...

  Inserted rows 90000 – 92000


  Inserted rows 92000 – 94000


  Inserted rows 94000 – 96000


  Inserted rows 96000 – 98000


  Inserted rows 98000 – 100000

Ingestion complete!


## 5. Validation — SQL Queries

In [6]:
# Row count
count_df = conn.query('SELECT COUNT(*) AS TOTAL FROM TRANSACTIONS')
print(f'Total rows in DB2: {count_df["TOTAL"].iloc[0]:,}')

# Fraud breakdown
fraud_df = conn.query('''
    SELECT FRAUDE, COUNT(*) AS CNT,
           CAST(COUNT(*) AS DOUBLE) / SUM(COUNT(*)) OVER () * 100 AS PCT
    FROM TRANSACTIONS
    GROUP BY FRAUDE
    ORDER BY FRAUDE
''')
print('\nClass distribution in DB2:')
print(fraud_df)

# Split breakdown
split_df = conn.query('''
    SELECT SPLIT, FRAUDE, COUNT(*) AS CNT
    FROM TRANSACTIONS
    GROUP BY SPLIT, FRAUDE
    ORDER BY SPLIT, FRAUDE
''')
print('\nTrain/test split:')
print(split_df)

Total rows in DB2: 100,000



Class distribution in DB2:
   FRAUDE    CNT     PCT
0       0  86265  86.265
1       1  13735  13.735



Train/test split:
   SPLIT  FRAUDE    CNT
0   test       0  17253
1   test       1   2747
2  train       0  69012
3  train       1  10988


In [7]:
# Read training set back into pandas (for local baseline)
train_from_db2 = conn.query("SELECT * FROM TRANSACTIONS WHERE SPLIT = 'train'")
print(f'Training set from DB2: {train_from_db2.shape}')
train_from_db2.head(3)

Training set from DB2: (80000, 13)


,VALOR_BRL,SALDO_ANTERIOR_PAGADOR,SALDO_ANTERIOR_RECEBEDOR,HORA_DIA,DIA_UTIL,HORARIO_NOTURNO,ACIMA_LIMITE_NOTURNO,RAZAO_SALDO_RESIDUAL,PROPORCAO_VALOR_RECEBEDOR,TIPO_TRANSACAO_ENC,DIA_SEMANA_ENC,FRAUDE,SPLIT
0,1011935.75,4.990342e+06,5530.55,15.0,0.0,0.0,0.0,0.797221,0.994564,0.0,6.0,0,train
1,1293676.13,1.294970e+06,28564.82,12.0,0.0,0.0,0.0,0.000999,0.978397,0.0,0.0,0,train
2,1735989.58,1.297753e+07,25703.68,16.0,0.0,0.0,0.0,0.866231,0.985410,0.0,0.0,0,train


## 6. Close Connection

In [8]:
conn.close()
print('DB2 connection closed.')

DB2 connection closed.
